In [ ]:
import json
import yfinance as yf
from openai import OpenAI

# 1. API Key Setup
# Set your key directly or ensure the OPENAI_API_KEY environment variable is set
api_key = "YOUR_OPENAI_API_KEY"

# 2. Initialize OpenAI Client
client = OpenAI(api_key=api_key)

# 3. Financial Helper Functions
def get_current_stock_price(ticker_symbol: str) -> str:
    """Fetch live stock price using Yahoo Finance."""
    try:
        stock = yf.Ticker(ticker_symbol)
        price = stock.fast_info["last_price"]
        currency = stock.fast_info.get("currency", "USD")
        return f"The current price of {ticker_symbol.upper()} is {price:.2f} {currency}."
    except Exception as e:
        return f"Could not retrieve current price for {ticker_symbol}: {str(e)}"

def get_5yr_avg_stock_price(ticker_symbol: str) -> str:
    """Fetch 5-year average closing price using Yahoo Finance."""
    try:
        stock = yf.Ticker(ticker_symbol)
        hist = stock.history(period="5y")
        if hist.empty:
            return f"No historical data found for {ticker_symbol.upper()}."
        avg_price = hist["Close"].mean()
        return f"The average closing price of {ticker_symbol.upper()} over the past 5 years is ${avg_price:.2f}."
    except Exception as e:
        return f"Could not retrieve 5-year average data for {ticker_symbol}: {str(e)}"

def get_financial_metrics(ticker_symbol: str) -> str:
    """Fetch key financial metrics (P/E, PEG, ROE, P/B, D/E)."""
    try:
        stock = yf.Ticker(ticker_symbol)
        info = stock.info
        
        pe_ratio = info.get("trailingPE", "N/A")
        peg_ratio = info.get("pegRatio", "N/A")
        roe = info.get("returnOnEquity", "N/A")
        pb_ratio = info.get("priceToBook", "N/A")
        de_ratio = info.get("debtToEquity", "N/A")
        
        formatted_roe = f"{roe * 100:.2f}%" if isinstance(roe, (int, float)) else "N/A"
        formatted_pe = f"{pe_ratio:.2f}" if isinstance(pe_ratio, (int, float)) else "N/A"
        formatted_peg = f"{peg_ratio:.2f}" if isinstance(peg_ratio, (int, float)) else "N/A"
        formatted_pb = f"{pb_ratio:.2f}" if isinstance(pb_ratio, (int, float)) else "N/A"
        formatted_de = f"{de_ratio:.2f}" if isinstance(de_ratio, (int, float)) else "N/A"

        return (
            f"Financial Metrics for {ticker_symbol.upper()}:\n"
            f"- Price-to-Earnings (P/E): {formatted_pe}\n"
            f"- Price/Earnings-to-Growth (PEG): {formatted_peg}\n"
            f"- Return on Equity (ROE): {formatted_roe}\n"
            f"- Price-to-Book (P/B): {formatted_pb}\n"
            f"- Debt-to-Equity (D/E): {formatted_de}"
        )
    except Exception as e:
        return f"Could not retrieve financial metrics for {ticker_symbol}: {str(e)}"

def evaluate_buy_decision(
    ticker_symbol: str, 
    max_pe: float = 25.0, 
    max_peg: float = 1.5, 
    min_roe: float = 0.15, 
    max_de: float = 150.0
) -> str:
    """Evaluates whether to buy a stock using custom or recommended thresholds."""
    try:
        stock = yf.Ticker(ticker_symbol)
        info = stock.info
        
        pe = info.get("trailingPE")
        peg = info.get("pegRatio")
        roe = info.get("returnOnEquity")
        de = info.get("debtToEquity")
        
        score = 0
        total_checks = 0
        reasons = []

        # 1. P/E Check
        if pe is not None:
            total_checks += 1
            if pe <= max_pe:
                score += 1
                reasons.append(f"P/E ratio is {pe:.2f} (Target: <= {max_pe}). Pass.")
            else:
                reasons.append(f"P/E ratio is high at {pe:.2f} (Target: <= {max_pe}). Fail.")

        # 2. PEG Check
        if peg is not None:
            total_checks += 1
            if peg <= max_peg:
                score += 1
                reasons.append(f"PEG ratio is {peg:.2f} (Target: <= {max_peg}). Pass.")
            else:
                reasons.append(f"PEG ratio is high at {peg:.2f} (Target: <= {max_peg}). Fail.")

        # 3. ROE Check
        if roe is not None:
            total_checks += 1
            if roe >= min_roe:
                score += 1
                reasons.append(f"ROE is {roe*100:.2f}% (Target: >= {min_roe*100:.2f}%). Pass.")
            else:
                reasons.append(f"ROE is low at {roe*100:.2f}% (Target: >= {min_roe*100:.2f}%). Fail.")

        # 4. Debt-to-Equity Check
        if de is not None:
            total_checks += 1
            if de <= max_de:
                score += 1
                reasons.append(f"Debt-to-Equity is {de:.2f} (Target: <= {max_de}). Pass.")
            else:
                reasons.append(f"Debt-to-Equity is high at {de:.2f} (Target: <= {max_de}). Fail.")

        if total_checks == 0:
            return f"Insufficient metrics data available for {ticker_symbol.upper()}."

        pass_rate = score / total_checks
        if pass_rate >= 0.75:
            verdict = "BUY / BULLISH"
        elif pass_rate >= 0.50:
            verdict = "HOLD / NEUTRAL"
        else:
            verdict = "SELL / BEARISH"

        return (
            f"=== Evaluation Report for {ticker_symbol.upper()} ===\n"
            f"Overall Recommendation: {verdict}\n"
            f"Criteria Passed: {score}/{total_checks}\n\n"
            f"Detailed Breakdown:\n" + "\n".join(f"- {r}" for r in reasons)
        )

    except Exception as e:
        return f"Error executing valuation evaluation: {str(e)}"

# 4. Interactive Metrics Configuration Function
def prompt_user_for_metrics() -> dict:
    """Asks user whether to use defaults or custom target thresholds."""
    print("\n--- Step 2: Metric Strategy Selection ---")
    print("1. Use AI Recommended Metrics (P/E <= 25, PEG <= 1.5, ROE >= 15%, D/E <= 150)")
    print("2. Use My Own Metric Thresholds")
    
    choice = input("\nSelect choice (1 or 2): ").strip()
    
    if choice == "2":
        try:
            max_pe = float(input("Enter Max acceptable P/E ratio [default 25]: ") or 25.0)
            max_peg = float(input("Enter Max acceptable PEG ratio [default 1.5]: ") or 1.5)
            min_roe = float(input("Enter Min acceptable ROE % [default 15]: ") or 15.0) / 100.0
            max_de = float(input("Enter Max acceptable Debt-to-Equity [default 150]: ") or 150.0)
            
            print("Custom metric targets saved.\n")
            return {
                "max_pe": max_pe,
                "max_peg": max_peg,
                "min_roe": min_roe,
                "max_de": max_de
            }
        except ValueError:
            print("Invalid input detected. Falling back to AI recommended values.\n")
    
    print("Using AI Recommended Financial Metrics.\n")
    return {"max_pe": 25.0, "max_peg": 1.5, "min_roe": 0.15, "max_de": 150.0}

# 5. Tool Schemas
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_stock_price",
            "description": "Fetch current stock price.",
            "parameters": {
                "type": "object",
                "properties": {"ticker_symbol": {"type": "string"}},
                "required": ["ticker_symbol"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_5yr_avg_stock_price",
            "description": "Fetch 5-year average price.",
            "parameters": {
                "type": "object",
                "properties": {"ticker_symbol": {"type": "string"}},
                "required": ["ticker_symbol"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_financial_metrics",
            "description": "Fetch current financial metrics of a stock.",
            "parameters": {
                "type": "object",
                "properties": {"ticker_symbol": {"type": "string"}},
                "required": ["ticker_symbol"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "evaluate_buy_decision",
            "description": "Evaluate if a stock should be bought based on target criteria.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker_symbol": {"type": "string", "description": "Stock ticker symbol."}
                },
                "required": ["ticker_symbol"],
            },
        },
    }
]

# 6. Agent Execution Routine
def run_agent(user_prompt: str, user_thresholds: dict):
    messages = [{"role": "user", "content": user_prompt}]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    response_message = response.choices[0].message

    if response_message.tool_calls:
        messages.append(response_message)

        for tool_call in response_message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            name = tool_call.function.name
            
            if name == "get_current_stock_price":
                tool_result = get_current_stock_price(args["ticker_symbol"])
            elif name == "get_5yr_avg_stock_price":
                tool_result = get_5yr_avg_stock_price(args["ticker_symbol"])
            elif name == "get_financial_metrics":
                tool_result = get_financial_metrics(args["ticker_symbol"])
            elif name == "evaluate_buy_decision":
                tool_result = evaluate_buy_decision(
                    args["ticker_symbol"],
                    **user_thresholds
                )
            else:
                tool_result = "Tool not found."

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result,
                }
            )

        final_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages
        )
        return final_response.choices[0].message.content

    return response_message.content

# 7. Main Continuous Loop
if __name__ == "__main__":
    print("=" * 60)
    print("      AI STOCK EVALUATION AGENT")
    print("=" * 60)

    while True:
        # STEP 1: Gather Stock Ticker Symbols
        raw_tickers = input("\n[Step 1] Enter ticker symbol(s) separated by commas (or 'quit' to exit): ")
        
        # Check if user wants to exit early
        if raw_tickers.strip().lower() in ["quit", "exit"]:
            print("Exiting Stock Evaluation Agent. Goodbye!")
            break

        tickers = [t.strip().upper() for t in raw_tickers.split(",") if t.strip()]

        if not tickers:
            print("No tickers entered. Please try again.")
            continue

        # STEP 2: Configure Evaluation Metrics
        thresholds = prompt_user_for_metrics()

        # STEP 3: Execute Buy Evaluations & Output Verdicts
        print("=" * 60)
        print("--- Step 3: Running AI Financial Evaluations ---")
        print("=" * 60)

        for ticker in tickers:
            prompt = f"Evaluate whether {ticker} is a good stock to buy right now."
            print(f"\nAnalyzing {ticker}...")
            
            agent_response = run_agent(prompt, thresholds)
            
            print("-" * 50)
            print(f"Agent Verdict for {ticker}:\n")
            print(agent_response)
            print("-" * 50)

        # Prompt at the end of the evaluation run
        print("\n" + "=" * 60)
        user_choice = input("Is there anything else you would like to analyze? (yes/no): ").strip().lower()
        
        if user_choice not in ["y", "yes"]:
            print("Thank you for using the Stock Decision Agent. Have a great day!")
            break